In [1]:
import os
import warnings
import seaborn as sns
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt

In [ ]:
# Define root directory
root = '.'

df = pd.read_csv('./dataset/maison-llf-features.CSV', sep=",")  ### maison-llf-features_TEST.CSV

ana = pd.read_csv('./dataset/maison-llf-demographics.CSV', sep=",")  ### maison-llf-demographics_TEST

In [5]:
ana_col = list(ana.columns)

ana_encoded = ana[["participant", "age", "sex", "education", "work", "fracture-type", "ethnicity"]].copy()

# male=1, female=0
ana_encoded["sex_male"] = ana_encoded["sex"].map({"male": 1, "female": 0})

# Education label encoding with doctorate as highest level
education_order = {
    "secondary education": 0,
    "undergraduate degree": 1,
    "graduate degree": 2,
    "doctorate degree": 3
}
ana_encoded["education_label"] = ana_encoded["education"].map(education_order)

# retired=0, employed part-time=1
ana_encoded["work_part_time"] = ana_encoded["work"].map({"retired": 0, "employed part-time": 1})

fracture_dummies = pd.get_dummies(   ### One-Hot Encoding.
    ana_encoded["fracture-type"],
    prefix="fracture",
    dtype=int
)

ethnicity_dummies = pd.get_dummies(   ### One-Hot Encoding.
    ana_encoded["ethnicity"],
    prefix="ethnicity",
    dtype=int
)

In [6]:
#fracture_dummies

In [7]:
### finisco di creare l'anagrafica
num_ana = pd.concat(
    [
        ana_encoded[["participant", "age", "sex_male", "education_label", "work_part_time"]],
        fracture_dummies,
        ethnicity_dummies,
    ],
    axis=1,
)

In [8]:
### join con il dataset dei records

data = df.merge(num_ana, how='right', on='participant')


In [9]:
# Compute quartiles for discretization
siss_q1, siss_q2, siss_q3 = np.percentile(data["sis"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(data["ohs"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(data["oks"], [25, 50, 75])

# Define quartile-based bins and labels
quartile_labels = [0, 1, 2, 3]

# Apply discretization
data["SISS_Category_Q"] = pd.cut(data["sis"], bins=[data["sis"].min(), siss_q1, siss_q2, siss_q3, data["sis"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)
data["OHSS_Category_Q"] = pd.cut(data["ohs"], bins=[data["ohs"].min(), ohss_q1, ohss_q2, ohss_q3, data["ohs"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

data["OKSS_Category_Q"] = pd.cut(data["oks"], bins=[data["oks"].min(), okss_q1, okss_q2, okss_q3, data["oks"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

In [10]:
# Extract only numeric features for LOPO (drop timestamps/string columns).
exclude_cols = [
    "participant",
    "timestamp",
    "clinical-timestamp",
    #"motion-max-timestamp",
    #"step-max-timestamp",
    "SISS_Category_Q",
    "OHSS_Category_Q",
    "OKSS_Category_Q",
]

feature_cols = [c for c in data.columns if c not in exclude_cols]
X = data[feature_cols].select_dtypes(include=[np.number]).copy()
groups = data["participant"]

In [ ]:
# Save the processed DataFrame to CSV
data.to_csv('./dataset/dataset_for_EDA.csv', index=False)